# ✈ AeroShield — SQL Loader
**Layer 2 — Load Python outputs into PostgreSQL**

This notebook reads the CSV files exported by `01_aeroshield_pipeline.ipynb`
and loads them into a PostgreSQL database using SQLAlchemy.

---
### Prerequisites
1. Run `01_aeroshield_pipeline.ipynb` first to generate the output CSVs
2. Create a `.env` file in the project root (copy from `.env.example`)
3. Ensure PostgreSQL is running and the `aeroshield` database exists

```sql
-- Run this once in pgAdmin or psql:
CREATE DATABASE aeroshield;
```

In [1]:
# Install if needed:
#!pip install python-dotenv sqlalchemy psycopg2-binary pandas

In [2]:
import pandas as pd
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# ── Load credentials from .env (never hardcode these) ────────────────────────
load_dotenv()

DB_HOST     = os.getenv("DB_HOST",     "localhost")
DB_PORT     = os.getenv("DB_PORT",     "5432")
DB_NAME     = os.getenv("DB_NAME",     "aeroshield")
DB_USER     = os.getenv("DB_USER",     "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD")      # no default — must be in .env
OUTPUT_DIR  = os.getenv("OUTPUT_DIR",  "./outputs")

if not DB_PASSWORD:
    raise EnvironmentError(
        "DB_PASSWORD not found. Create a .env file with DB_PASSWORD=your_password."
    )

# ── Build connection string ────────────────────────────────────────────────────
conn_str = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine   = create_engine(conn_str)

# Test connection
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print(f"✅ Connected to PostgreSQL → {DB_HOST}:{DB_PORT}/{DB_NAME}")

✅ Connected to PostgreSQL → localhost:5432/aeroshield


In [3]:
# =============================================================================
# Load all summary CSVs into PostgreSQL
# =============================================================================

files = {
    "airport_summary.csv"       : "airport_summary",
    "route_summary.csv"         : "route_summary",
    "carrier_summary.csv"       : "carrier_summary",
    "monthly_summary.csv"       : "monthly_summary",
    "delay_cause_summary.csv"   : "delay_cause_summary",
    "model_comparison.csv"      : "model_comparison",
    "rf_feature_importance.csv" : "rf_feature_importance",
}

for fname, table in files.items():
    fpath = os.path.join(OUTPUT_DIR, fname)

    if not os.path.exists(fpath):
        print(f"  ⚠️  Skipped (file not found): {fname}")
        continue

    df = pd.read_csv(fpath)
    df.columns = df.columns.str.lower()          # lowercase for PostgreSQL convention
    df.to_sql(table, engine, if_exists="replace", index=False)
    print(f"  ✅ Loaded {table:30s} — {len(df):,} rows")

print("\n🎉 All tables loaded into PostgreSQL!")
print("    Next step → Run aeroshield_sql.sql to create analytical views")

  ✅ Loaded airport_summary                — 346 rows
  ✅ Loaded route_summary                  — 6,630 rows
  ✅ Loaded carrier_summary                — 14 rows
  ✅ Loaded monthly_summary                — 6 rows
  ✅ Loaded delay_cause_summary            — 5 rows
  ✅ Loaded model_comparison               — 2 rows
  ✅ Loaded rf_feature_importance          — 20 rows

🎉 All tables loaded into PostgreSQL!
    Next step → Run aeroshield_sql.sql to create analytical views


In [4]:
# =============================================================================
# Quick verification — list tables and row counts
# =============================================================================

verify_query = """
SELECT table_name,
       (xpath('/row/cnt/text()',
              query_to_xml(format('SELECT COUNT(*) AS cnt FROM %I', table_name),
                           false, true, '')))[1]::text::int AS row_count
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

with engine.connect() as conn:
  result = pd.read_sql(text(verify_query), conn)
print("Tables in aeroshield database:")
print(result.to_string(index=False))

Tables in aeroshield database:
           table_name  row_count
      airport_summary        346
      carrier_summary         14
  delay_cause_summary          5
      flights_cleaned          0
     model_comparison          2
      monthly_summary          6
rf_feature_importance         20
        route_summary       6630
